# Module 5 — Inverse Kinematics
## Unit 6 — Singularities and Solution Selection
### Lesson 6.3 — Choosing Among Solutions (Nearest, Smooth, Limit-Safe)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Pick the feasible solution nearest the current pose.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12, -L2*s12],[L1*c1+L2*c12, L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

L1,L2=0.4,0.3
def norm180(a): return (a+180)%360-180
def ang_dist(a,b):  # degrees, wrapped
    return abs(norm180(a-b))
def choose_solution(sols, theta_cur, L1, L2):
    best=None; bestcost=np.inf
    for (t1,t2) in sols:
        d=np.hypot(ang_dist(np.degrees(t1),np.degrees(theta_cur[0])),
                   ang_dist(np.degrees(t2),np.degrees(theta_cur[1])))
        if d<bestcost: bestcost=d; best=(t1,t2)
    return best,bestcost
sols=ik_2link_closed(0.5,0.0,L1,L2)
cur=np.radians([-30,80])
best,cost=choose_solution(sols,cur,L1,L2)
print("chosen:",tuple(np.round(np.degrees(best),2)),"cost(deg)=",round(cost,1))


## Coding Exercise (§8)
Nearest = elbow-down here; forcing it out → the other is chosen.


In [ ]:
# YOUR CODE HERE
sols=ik_2link_closed(0.5,0.0,L1,L2)
best,_=choose_solution(sols,np.radians([-30,80]),L1,L2)
assert best[1]>0                                   # elbow-down nearest
# if only elbow-up remains, it must be chosen
up=[s for s in sols if s[1]<0]
best2,_=choose_solution(up,np.radians([-30,80]),L1,L2)
assert best2[1]<0
print("selection OK")


## Check your work


In [ ]:
best,_=choose_solution(ik_2link_closed(0.5,0.0,L1,L2),np.radians([-30,80]),L1,L2)
assert abs(np.degrees(best[1])-90)<1e-6
print("All checks passed.")
